In [1]:
import pandas as pd
import numpy as np
import torchvision
import torch

In [2]:
summary = pd.read_csv("../model_summary.csv")

In [16]:
summary

,model_id,k_fold,frac_val,val_best_acc,val_best_auc,val_best_sp,val_best_sn,val_avg_acc,val_avg_auc,val_avg_sp,...,double_img,output_tab,backbone,early_start,timestamp,total_hours,average_cross_hours,torchvision_version,torch_version,history_added
0,32affbbcac12427a84e10af5e44db802,NaN,0.2,0.847134,0.790345,1.000000,0.826087,0.769618,0.691058,0.880811,...,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0
1,89b1209eca2e44eeb1b59061fa687789,NaN,0.2,0.821656,0.776048,1.000000,0.913043,0.725541,0.645374,0.839009,...,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0
2,0de11e06a2444858bcdfac8ebf136a7a,NaN,0.2,0.707006,0.530454,1.000000,1.000000,0.693949,0.500441,0.967838,...,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0
3,de2371ef299b4e109f4d8cc7b8a16d43,NaN,0.2,0.770701,0.690462,1.000000,1.000000,0.616752,0.550106,0.711081,...,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0
4,2681ca9e9e6f42c9830af1eedb7d52c6,NaN,0.2,0.757962,0.646396,1.000000,1.000000,0.672675,0.515948,0.894505,...,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,7108c95a38904af095098bbd5d999d4f,NaN,0.2,0.859873,0.791911,0.990991,0.847826,0.770701,0.708945,0.858108,...,0.0,0.0,regnet16x,100.0,2022-11-20 16:46:38.043204,02:04:19.17,02:04:19.17,0.13.0+cu113,1.12.0+cu113,0.0
837,16eb45c48ac34fb39a00bbb17c2f6e45,NaN,0.2,0.840764,0.793772,0.945946,0.956522,0.774522,0.701910,0.877297,...,0.0,0.0,regnet16x,100.0,2022-11-20 18:49:29.798087,02:02:47.14,02:02:47.14,0.13.0+cu113,1.12.0+cu113,0.0
838,344a717aeec64b46a3a5f7ac5a2fe59c,NaN,0.2,0.859873,0.828241,0.954955,0.913043,0.785096,0.720717,0.876216,...,0.0,0.0,regnet16x,100.0,2022-11-20 18:51:39.825998,02:04:57.14,02:04:57.14,0.13.0+cu113,1.12.0+cu113,0.0
839,a9ec920847fc4dd4ade79a003c4891dc,NaN,0.2,0.834395,0.778398,0.954955,0.717391,0.777834,0.714309,0.867748,...,0.0,0.0,regnet16x,100.0,2022-11-20 20:56:27.015538,02:04:42.51,02:04:42.51,0.13.0+cu113,1.12.0+cu113,0.0


In [8]:
summary = summary.drop_duplicates(subset=["model_id"])
summary.shape

(673, 35)

In [9]:
summary.columns

Index(['model_id', 'k_fold', 'frac_val', 'val_best_acc', 'val_best_auc',
       'val_best_sp', 'val_best_sn', 'val_avg_acc', 'val_avg_auc',
       'val_avg_sp', 'val_avg_sn', 'train_best_acc', 'train_best_auc',
       'train_best_sp', 'train_best_sn', 'train_avg_acc', 'train_avg_auc',
       'train_avg_sp', 'train_avg_sn', 'host_name', 'is_inception', 'optim',
       'lr', 'epochs', 'double_img', 'output_tab', 'backbone',
       'feature_extract', 'early_start', 'timestamp', 'total_hours',
       'average_cross_hours', 'torchvision_version', 'torch_version',
       'history_added'],
      dtype='object')

In [11]:
summary.drop(columns=["feature_extract","is_inception"],inplace=True)

In [14]:
summary["output_tab"] = summary["output_tab"].fillna(0)
summary["output_tab"]

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
836    0.0
837    0.0
838    0.0
839    0.0
840    0.0
Name: output_tab, Length: 673, dtype: float64

In [30]:
summary["shap_val"] = 0
summary["shap_oos"] = 0
summary["pred_val"] = 0
summary["pred_oos"] = 0
summary["batch_size"] = 16

In [31]:
summary

,model_id,k_fold,frac_val,val_best_acc,val_best_auc,val_best_sp,val_best_sn,val_avg_acc,val_avg_auc,val_avg_sp,val_avg_sn,train_best_acc,train_best_auc,train_best_sp,train_best_sn,train_avg_acc,train_avg_auc,train_avg_sp,train_avg_sn,host_name,optim,lr,epochs,double_img,output_tab,backbone,early_start,timestamp,total_hours,average_cross_hours,torchvision_version,torch_version,history_added,shap_val,shap_oos,pred_val,pred_oos,batch_size
0,32affbbcac12427a84e10af5e44db802,NaN,0.2,0.847134,0.790345,1.000000,0.826087,0.769618,0.691058,0.880811,0.501304,0.946755,0.922611,0.976959,0.874251,0.872263,0.822313,0.934747,0.709880,magneto,sgd,0.0100,100.0,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
1,89b1209eca2e44eeb1b59061fa687789,NaN,0.2,0.821656,0.776048,1.000000,0.913043,0.725541,0.645374,0.839009,0.451739,0.843594,0.762769,0.953917,0.580838,0.801514,0.703370,0.924286,0.482455,magneto,ranger,0.0100,100.0,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
2,0de11e06a2444858bcdfac8ebf136a7a,NaN,0.2,0.707006,0.530454,1.000000,1.000000,0.693949,0.500441,0.967838,0.033043,0.722130,0.508016,1.000000,0.221557,0.718752,0.499651,0.992834,0.006467,magneto,adam,0.0100,100.0,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
3,de2371ef299b4e109f4d8cc7b8a16d43,NaN,0.2,0.770701,0.690462,1.000000,1.000000,0.616752,0.550106,0.711081,0.389130,0.787022,0.669714,1.000000,0.443114,0.741414,0.587601,0.933825,0.241377,magneto,radam,0.0100,100.0,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
4,2681ca9e9e6f42c9830af1eedb7d52c6,NaN,0.2,0.757962,0.646396,1.000000,1.000000,0.672675,0.515948,0.894505,0.137391,0.768719,0.650135,1.000000,0.413174,0.721248,0.528271,0.962650,0.093892,magneto,radam,0.0100,100.0,0.0,0.0,mobile,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,7108c95a38904af095098bbd5d999d4f,NaN,0.2,0.859873,0.791911,0.990991,0.847826,0.770701,0.708945,0.858108,0.559783,0.910150,0.875162,0.958525,0.796407,0.859418,0.799292,0.934631,0.663952,magneto,radam,0.0010,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 16:46:38.043204,02:04:19.17,02:04:19.17,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
837,16eb45c48ac34fb39a00bbb17c2f6e45,NaN,0.2,0.840764,0.793772,0.945946,0.956522,0.774522,0.701910,0.877297,0.526522,0.971714,0.960153,0.986175,0.934132,0.890865,0.855529,0.935069,0.775988,magneto,ranger,0.0001,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 18:49:29.798087,02:02:47.14,02:02:47.14,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
838,344a717aeec64b46a3a5f7ac5a2fe59c,NaN,0.2,0.859873,0.828241,0.954955,0.913043,0.785096,0.720717,0.876216,0.565217,0.976705,0.967294,0.988479,0.946108,0.906040,0.875024,0.944839,0.805210,magneto,radam,0.0001,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 18:51:39.825998,02:04:57.14,02:04:57.14,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16
839,a9ec920847fc4dd4ade79a003c4891dc,NaN,0.2,0.834395,0.778398,0.954955,0.717391,0.777834,0.714309,0.867748,0.560870,0.958403,0.946101,0.981567,0.922156,0.908602,0.875491,0.950023,0.800958,magneto,ranger,0.0005,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 20:56:27.015538,02:04:42.51,02:04:42.51,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0,16


In [32]:
summary.to_csv("../model_summary.csv", index=False)

In [21]:
summary["backbone"].tail()

836    regnet16x
837    regnet16x
838    regnet16x
839    regnet16x
840    regnet16x
Name: backbone, dtype: object

In [22]:
summary

,model_id,k_fold,frac_val,val_best_acc,val_best_auc,val_best_sp,val_best_sn,val_avg_acc,val_avg_auc,val_avg_sp,...,timestamp,total_hours,average_cross_hours,torchvision_version,torch_version,history_added,shap_val,shap_oos,pred_val,pred_oos
0,32affbbcac12427a84e10af5e44db802,NaN,0.2,0.847134,0.790345,1.000000,0.826087,0.769618,0.691058,0.880811,...,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
1,89b1209eca2e44eeb1b59061fa687789,NaN,0.2,0.821656,0.776048,1.000000,0.913043,0.725541,0.645374,0.839009,...,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
2,0de11e06a2444858bcdfac8ebf136a7a,NaN,0.2,0.707006,0.530454,1.000000,1.000000,0.693949,0.500441,0.967838,...,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
3,de2371ef299b4e109f4d8cc7b8a16d43,NaN,0.2,0.770701,0.690462,1.000000,1.000000,0.616752,0.550106,0.711081,...,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
4,2681ca9e9e6f42c9830af1eedb7d52c6,NaN,0.2,0.757962,0.646396,1.000000,1.000000,0.672675,0.515948,0.894505,...,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,7108c95a38904af095098bbd5d999d4f,NaN,0.2,0.859873,0.791911,0.990991,0.847826,0.770701,0.708945,0.858108,...,2022-11-20 16:46:38.043204,02:04:19.17,02:04:19.17,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
837,16eb45c48ac34fb39a00bbb17c2f6e45,NaN,0.2,0.840764,0.793772,0.945946,0.956522,0.774522,0.701910,0.877297,...,2022-11-20 18:49:29.798087,02:02:47.14,02:02:47.14,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
838,344a717aeec64b46a3a5f7ac5a2fe59c,NaN,0.2,0.859873,0.828241,0.954955,0.913043,0.785096,0.720717,0.876216,...,2022-11-20 18:51:39.825998,02:04:57.14,02:04:57.14,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
839,a9ec920847fc4dd4ade79a003c4891dc,NaN,0.2,0.834395,0.778398,0.954955,0.717391,0.777834,0.714309,0.867748,...,2022-11-20 20:56:27.015538,02:04:42.51,02:04:42.51,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0


In [24]:
summary["early_start"]

0      100.0
1      100.0
2      100.0
3      100.0
4      100.0
       ...  
836    100.0
837    100.0
838    100.0
839    100.0
840    100.0
Name: early_start, Length: 673, dtype: float64

In [27]:
pd.options.display.max_columns = None

summary[summary["backbone"] == "regnet16x"]

,model_id,k_fold,frac_val,val_best_acc,val_best_auc,val_best_sp,val_best_sn,val_avg_acc,val_avg_auc,val_avg_sp,val_avg_sn,train_best_acc,train_best_auc,train_best_sp,train_best_sn,train_avg_acc,train_avg_auc,train_avg_sp,train_avg_sn,host_name,optim,lr,epochs,double_img,output_tab,backbone,early_start,timestamp,total_hours,average_cross_hours,torchvision_version,torch_version,history_added,shap_val,shap_oos,pred_val,pred_oos
503,e8a69d58de8249ac8b7541edb480acd9,NaN,0.2,0.847134,0.804642,1.000000,0.782609,0.789618,0.716850,0.892613,0.541087,0.966722,0.951171,0.986175,0.922156,0.890349,0.840325,0.952926,0.727725,solid,sgd,0.0010,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
504,4b36c2f456264738a3aff859f0e6ab63,NaN,0.2,0.840764,0.784763,0.981982,0.891304,0.767389,0.693364,0.872162,0.514565,0.883527,0.830942,0.956221,0.712575,0.822047,0.735988,0.929700,0.542275,solid,adam,0.0010,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
505,01f2614dfc134b7cb98b970863a29ea2,NaN,0.2,0.770701,0.678711,1.000000,0.456522,0.726943,0.586024,0.926396,0.245652,0.841930,0.765301,1.000000,0.592814,0.771231,0.622208,0.957650,0.286766,solid,sgd,0.0001,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
506,c9d73b9f038d4a20835b8d3f9c210a4f,NaN,0.2,0.847134,0.809146,0.963964,0.782609,0.793503,0.737165,0.873243,0.601087,0.976705,0.970977,0.986175,0.958084,0.925574,0.897317,0.960922,0.833713,solid,adam,0.0001,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
507,5cb27a3c546441e183973b6a8f370b47,NaN,0.2,0.828025,0.770172,1.000000,0.695652,0.782484,0.693855,0.907928,0.479783,0.940100,0.925370,0.990783,0.892216,0.859983,0.789184,0.948548,0.629820,solid,sgd,0.0005,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
508,9a6d117bacc44ff5bc2edc3fe90019de,NaN,0.2,0.847134,0.791911,1.000000,0.782609,0.781720,0.725713,0.860991,0.590435,0.938436,0.903957,0.990783,0.844311,0.877937,0.825745,0.943226,0.708263,solid,adam,0.0005,100.0,0.0,0.0,regnet16x,100.0,NaN,NaN,NaN,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
831,9ac37b0f79fc4165b1e5166e731171b2,NaN,0.2,0.828025,0.745006,1.000000,0.847826,0.750000,0.656179,0.882793,0.429565,0.836938,0.746647,0.960829,0.550898,0.787171,0.670636,0.932949,0.408323,magneto,ranger,0.0100,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 14:41:33.415158,04:32:32.62,04:32:32.62,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
832,80e90a0e873343ac988ea94dabee1f16,NaN,0.2,0.834395,0.794849,0.981982,0.891304,0.763185,0.704394,0.846396,0.562391,0.946755,0.922839,0.979263,0.880240,0.872895,0.821351,0.937373,0.705329,magneto,sgd,0.0100,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 14:41:43.565423,04:32:42.77,04:32:42.77,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
833,133be341743f4d0d8700ed74d7da0c08,NaN,0.2,0.757962,0.628672,1.000000,0.913043,0.691338,0.532074,0.916757,0.147391,0.760399,0.601555,1.000000,0.263473,0.731830,0.549284,0.960184,0.138383,magneto,radam,0.0100,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 14:42:10.866159,04:33:10.07,04:33:10.07,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
834,9f6d9dc358c64d6586520edfed74d1b1,NaN,0.2,0.707006,0.521152,1.000000,0.826087,0.703567,0.500177,0.991441,0.008913,0.722130,0.516322,1.000000,0.161677,0.721531,0.500230,0.998364,0.002096,magneto,adam,0.0100,100.0,0.0,0.0,regnet16x,100.0,2022-11-20 14:42:14.183464,04:33:13.39,04:33:13.39,0.13.0+cu113,1.12.0+cu113,0.0,0,0,0,0
